# Parameter tuning with GridSearchCV to see if we can easily improve results

# Lets see how our initial models do now that we have a lot more data

We already went through this so I will quickly do it below

While doing research, we learned that we could automatically tune model parameters using GridSearchCV from sklearn. This allows us to specify a grid of parameters to search over, and the model will be trained and evaluated for each combination of parameters in the grid. This can help us find the best set of parameters for our models and potentially improve their performance, without us having to do anything.

So we will see if this can make any improvements before we continue.

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import GridSearchCV

In [4]:
df = pd.read_csv("data/extra_and_merged_data.csv", index_col="date", parse_dates=True)
df.sort_index(inplace=True)

In [5]:
X = df[["summary", "pct_change", "volume"]]
y = df["target"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [7]:
print(f"Training Range: {X_train.index.min()} to {X_train.index.max()}")
print(f"Testing Range:  {X_test.index.min()} to {X_test.index.max()}")

Training Range: 2003-02-20 00:00:00 to 2018-03-23 00:00:00
Testing Range:  2018-03-26 00:00:00 to 2021-12-30 00:00:00


In [8]:
text_features = "summary"
numerical_features = ["pct_change", "volume"]

In [9]:


preprocessor_sgd = ColumnTransformer(
    transformers=[
        (
            "text",
            TfidfVectorizer(max_features=5000, ngram_range=(1, 2)),
            text_features,
        ),
        ("num", StandardScaler(), numerical_features),
    ]
)

preprocessor_lda = ColumnTransformer(
    transformers=[
        # Reduce text to 50 components using SVD (PCA for text)
        (
            "text",
            Pipeline(
                [
                    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
                    ("svd", TruncatedSVD(n_components=50)),
                ]
            ),
            text_features,
        ),
        ("num", StandardScaler(), numerical_features),
    ]
)

preprocessor_mnb = ColumnTransformer(
    transformers=[
        (
            "tfidf",
            TfidfVectorizer(max_features=5000, ngram_range=(1, 2)),
            text_features,
        ),
        ("scaler", MinMaxScaler(clip=True), numerical_features),
    ]
)

In [10]:
pipeline_sgd = Pipeline([
    ('prep', preprocessor_sgd),
    ('clf', SGDClassifier(loss='log_loss', penalty='l2', max_iter=1000, random_state=42))
])

pipeline_lda = Pipeline([
    ('prep', preprocessor_lda),
    ('clf', LinearDiscriminantAnalysis())
])

pipeline_mnb = Pipeline([
    ('prep', preprocessor_mnb),
    ('clf', MultinomialNB(alpha=0.1, fit_prior=False))
])


I am using gridsearch to find the best parameters for each model

In [ ]:
grid_sgd = GridSearchCV(
    estimator=pipeline_sgd,
    param_grid={
        "prep__text__max_features": [3000, 5000, 8000],
        "prep__text__ngram_range": [(1,1), (1,2), (1,3), (1,4), (1,5)],
        "clf__alpha": [1e-4, 1e-3, 1e-2],
        "clf__loss": ["hinge", "log_loss"],
        "clf__penalty": ["l2", "elasticnet"],
        "clf__learning_rate": ["optimal", "adaptive"],
        "clf__max_iter": [1000, 2000],
    },
    cv=3,
    n_jobs=-1,
    scoring="accuracy",
    verbose=1
)

grid_sgd.fit(X_train, y_train)

grid_lda = GridSearchCV(
    estimator=pipeline_lda,
    param_grid = {

    # TEXT FEATURE TUNING
    "prep__text__tfidf__max_features": [3000, 5000, 8000],
    "prep__text__tfidf__ngram_range": [(1,1), (1,2), (1,3), (1,4), (1,5)],

    # SVD REDUCTION TUNING (Pipeline part)
    "prep__text__svd__n_components": [20, 50, 100],

    # LDA MODEL TUNING
    "clf__solver": ["svd", "lsqr"],
    "clf__shrinkage": [None, "auto", 0.1, 0.5],
    },
    cv=3,
    n_jobs=-1,
    scoring="accuracy"
)
grid_lda.fit(X_train, y_train)

grid_mnb = GridSearchCV(
    estimator=pipeline_mnb,
    param_grid = {

    # TEXT FEATURE TUNING
    "prep__tfidf__max_features": [3000, 5000, 8000],
    "prep__tfidf__ngram_range": [(1,1), (1,2), (1,3), (1,4), (1,5)],

    # NAIVE BAYES TUNING
    "clf__alpha": [0.01, 0.1, 0.5, 1.0],
    "clf__fit_prior": [True, False],
},
    cv=3,
    n_jobs=-1,
    scoring="accuracy"
)
grid_mnb.fit(X_train, y_train)

Fitting 3 folds for each of 720 candidates, totalling 2160 fits


In [27]:
print("SGD Best Params:", grid_sgd.best_params_)
print("SGD Test Acc:", grid_sgd.score(X_test, y_test))

y_pred_sgd = grid_sgd.predict(X_test)

print("SGD Classification Report:")
print(confusion_matrix(y_test, y_pred_sgd))
print(classification_report(y_test, y_pred_sgd))

SGD Best Params: {'clf__alpha': 0.001, 'clf__loss': 'hinge'}
SGD Test Acc: 0.5637513171759747
SGD Classification Report:
[[  7 406]
 [  8 528]]
              precision    recall  f1-score   support

         0.0       0.47      0.02      0.03       413
         1.0       0.57      0.99      0.72       536

    accuracy                           0.56       949
   macro avg       0.52      0.50      0.38       949
weighted avg       0.52      0.56      0.42       949



In [26]:
print("LDA Best Params:", grid_lda.best_params_)
print("LDA Test Acc:", grid_lda.score(X_test, y_test))

y_pred_lda = grid_lda.predict(X_test)

print("LDA Classification Report:")
print(confusion_matrix(y_test, y_pred_lda))
print(classification_report(y_test, y_pred_lda))

LDA Best Params: {'clf__solver': 'lsqr'}
LDA Test Acc: 0.5553213909378293
LDA Classification Report:
[[ 68 345]
 [ 77 459]]
              precision    recall  f1-score   support

         0.0       0.47      0.16      0.24       413
         1.0       0.57      0.86      0.69       536

    accuracy                           0.56       949
   macro avg       0.52      0.51      0.46       949
weighted avg       0.53      0.56      0.49       949



In [25]:

print("MNB Best Params:", grid_mnb.best_params_)
print("MNB Test Acc:", grid_mnb.score(X_test, y_test))

y_pred_mnb = grid_mnb.predict(X_test)

print("MNB Classification Report:")
print(confusion_matrix(y_test, y_pred_mnb))
print(classification_report(y_test, y_pred_mnb))

MNB Best Params: {'clf__alpha': 1.0}
MNB Test Acc: 0.47418335089567965
MNB Classification Report:
[[238 175]
 [324 212]]
              precision    recall  f1-score   support

         0.0       0.42      0.58      0.49       413
         1.0       0.55      0.40      0.46       536

    accuracy                           0.47       949
   macro avg       0.49      0.49      0.47       949
weighted avg       0.49      0.47      0.47       949



In [ ]:
%%capture
%pip install joblib

In [ ]:
# exporting the baseline model to compare after we tune it
import joblib

joblib.dump(pipeline_sgd, "models/sgd_baseline_more_data.pkl")
joblib.dump(pipeline_lda, "models/lda_baseline_more_data.pkl")
joblib.dump(pipeline_mnb, "models/mnb_baseline_more_data.pkl")

['models/lda_baseline_more_data.pkl']

the models LDA have a pitfall which is causing some underlying issues. The model pretty much always guess up as they have noticed the trend of it rising. It has learned that guessing up is safer. the Naives Bayes came out to worse than random guessing. SGD has same issue of LDA but slightly more balance on its up vs down guesses but it is still heavily guessing up.

Our low F1-score is proving that the model is not actually learning market mechanics it is just guessing based of the trend. We are also drowning out our financial data with our news being a majority. So to combat that we are going to use the news to create a sentiment value and then pass that and the finical data to a model. 